# Pose predictors

## File Handling
To run predictions a `RobotEnvironment` object and a `HeadsetData` object is needed, those can be loaded from folders or created.

### Creation of RobotEnvironment and HeadsetData
Those 2 datatypes can be created from an GatheredRobotData object and a .vrs file respectively.

In [1]:
%load_ext autoreload
%autoreload 2
print(__debug__)
from headset_data import *
from robot_environment import *

False
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
robot_data_folder_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/r7_small_aruco3"
vrs_file_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/r7_small_aruco3_5fps.vrs"


robot_data_from_disk = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env_from_robot_data = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data_from_disk,
        number_of_sampled_datapoints=10,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

headset_data_from_vrs = HeadsetData.from_vrs_file(vrs_file_location)

length of chosen indices 10
Generating point cloud...
Loading pretrained dinov2_vitg14 from torch hub


Using cache found in /home/wmarx/.cache/torch/hub/facebookresearch_dinov2_main


image shape before processing: (10, 720, 720, 3)
xyz images shape: (10, 518, 518, 3)
aligning pointclouds using ICP


100%|██████████| 9/9 [00:43<00:00,  4.86s/it]
[ProgressLogger][INFO]: 2026-06-09 14:16:18: Opening /home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/r7_small_aruco3_5fps.vrs...
[TelemetryLogger][INFO]: RecordFileReader::doOpenFile, diskfile: success, 64664903
[MultiRecordFileReader][DEBUG]: Opened file '/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/r7_small_aruco3_5fps.vrs' and assigned to reader #0
[VrsDataProvider][INFO]: streamId 214-1/camera-rgb activated
[VrsDataProvider][INFO]: Utc stream found: 285-1
[VrsDataProvider][INFO]: Fail to activate streamId 286-1


### Adding labels to HeadsetData
To add labels to the HeadsetData for accuracy evaluation it has to be joined with an GatheredRobotData object with some supported marker (e.g. aruco/charuco). 
It can then be saved and does not need rebounding for labels (rebounding is still possible), so GatheredRobotData becomes obsolete

In [3]:
# Labeling by using a robot_env_from_robot_data
labeled_headset_data = create_robot_bound_headset_data(
        headset_data = headset_data_from_vrs,
        robot_data = robot_data_from_disk
    )

### Saving and loading RobotEnvironments and HeadsetData
`RobotEnvironment` and `HeadsetData` both have file interfaces which can be used to load/save them from/to the disk

In [4]:
processed_datasets_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/processed_datasets"

labeled_headset_data.save(processed_datasets_location, new_name = "quickstart_headset_data")
robot_env_from_robot_data.save(processed_datasets_location, new_name = "quickstart_robot_data")

robot_env_from_disk = RobotEnvironment.from_folder(f"{processed_datasets_location}/quickstart_robot_data")
headset_data_from_disk = HeadsetData.from_folder(f"{processed_datasets_location}/quickstart_headset_data")

Output folder already exists, deleting it ...
Output folder already exists, deleting it ...


### Alternative: Creating from TU-München Dataset
Alternative they cam be created from a TU-München Dataset: https://cvg.cit.tum.de/data/datasets/rgbd-dataset

In [5]:
from load_from_tum import *

tum_rgbd_dataset_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = robot_environment_and_headset_data_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.01,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
)

KeyboardInterrupt: 

### Visualising Datasets
Robot environments and Headset datasets can be visualized in 3d

In [6]:
vis_robot_env, vis_headset, vis_both = False, False, True

if vis_robot_env:
    robot_env_from_disk.visualize_3d_data()
if vis_headset:
    headset_data_from_disk.visualize_3d_data()
if vis_both:
    visualize_robot_camera_environment_combo(robot_env=robot_env_from_disk, headset_data=headset_data_from_disk)

## Testing Predictors

In [7]:
from predictor_grader import *
from pose_pred_points import *
from pose_pred_points_lines import *
from pose_pred_points_ellipsoids import *

### Creating Predictors
Now an `PosePredictor` can be created. An `PosePredictor` instance is build upon an `RobotEnvironment` instance and can predict positions from headset-images.

In [16]:
chosen_headset_data = headset_data_from_disk
chosen_robot_env = robot_env_from_disk

# Simple Point only predictor
point_predictor = OnlyPointsPredictor(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=chosen_robot_env.robot_bgr_images,
        cam1_xyz_images=chosen_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue())
)

### Grading the Performance of an Initialised Predictor:

In [17]:
init_predictor_grade = PredictionOnDataset(
    predictor = point_predictor,
    headset_data = chosen_headset_data,
    number_consecutive_update_pose_calls = 0,
    number_retry = 1
)

init_predictor_grade.print_summary()

Success rate: 0.9 for 40 predictions
Avg. time per prediction: 321.9753412064165ms
est_base_t_cam subcomponent times:

extract and match wrapper call           321.969 ms
Avg. error: 234.2mm and 25.8°
Median. error: 107.6mm and 6.5°
ATE RMSE: 357.4mm and 46.0°
RTE RMSE: 319.2mm and 27.3°


The predictions can also be visualized in 3d

In [19]:
visualize_prediction = True
if visualize_prediction:
    init_predictor_grade.visualize_predictions(
        robot_env=chosen_robot_env,
        show_label=False
    )

### Grading of multiple uninitialised Predictors in an Environment
Predictors provide `get_creation_function` methods, which can be used to initialize one, to grade the creation behaviour and time. Their signatures are similar to the of `__init__`. `GradablePosePredictor` provided additional configuration options on how many retries per prediction if `update_pose` should be used etc.

In [20]:
# Creation of the Predictors
points_light_glue = GradablePosePredictor(
    creator= OnlyPointsPredictor.get_creation_function(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue())
    ),
    name="points_light_glue"
)

#points_lines_light_glue = GradablePosePredictor(
#    creator=LinePredictor.get_creation_function(
#        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
#    ),
#    name="points_lines_light_glue"
#)

ellipsoids_light_glue = GradablePosePredictor(
    creator=EllipsoidPredictor.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
    ),
    name="ellipsoids_light_glue"
)

Now those can be used to create a grader object for multiple `PosePredictor` variants.

In [22]:
# Initialising the grader:
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[points_light_glue, ellipsoids_light_glue],
    headset_data = chosen_headset_data,
    robot_env = chosen_robot_env,
)
grader.print_summary()

100%|██████████| 40/40 [00:12<00:00,  3.11it/s]


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

  8%|▊         | 3/40 [00:00<00:08,  4.57it/s]


0: 2880x2880 1 pen, 2 round objects, 2 tools, 1 plastic object, 4 legos, 79.2ms
Speed: 14.3ms preprocess, 79.2ms inference, 2.8ms postprocess per image at shape (1, 3, 2880, 2880)


 10%|█         | 4/40 [00:01<00:16,  2.15it/s]

...


0: 2880x2880 1 pen, 3 round objects, 3 tools, 1 plastic object, 2 legos, 78.6ms
Speed: 14.9ms preprocess, 78.6ms inference, 2.6ms postprocess per image at shape (1, 3, 2880, 2880)


 98%|█████████▊| 39/40 [00:28<00:00,  1.39it/s]


0: 2880x2880 1 pen, 3 round objects, 1 tool, 1 plastic object, 1 metal object, 1 lego, 78.5ms
Speed: 14.5ms preprocess, 78.5ms inference, 2.6ms postprocess per image at shape (1, 3, 2880, 2880)


100%|██████████| 40/40 [00:29<00:00,  1.37it/s]

name                           success ratio %      T/frame [ms]    avg t_err [mm]  avg r_err [deg] 

points_light_glue              90.00                322             234.2           25.8           
ellipsoids_light_glue          90.00                730             445.2           41.6           
